# AI Agent Demo — LangChain + MCP 天气服务

本 Notebook 演示 **LangChain Agent 通过 MCP（Model Context Protocol）调用外部工具** 的完整流程，覆盖两种情况：

| 场景 | MCP Server | 数据来源 | 是否需要 API Key |
|---|---|---|---|
| **场景一：自建** | `demo/weather_mcp_server.py`（本 Notebook 生成） | Mock 数据 | 否 |
| **场景二：开源** | `open-meteo-mcp` | Open-Meteo 真实气象 API | **否** |
| **场景二：开源** | `mcp-server-weather` | OpenWeather 真实气象 API | 是 |

---

## 一、MCP 是什么？和 `demo-langchain.py` 有什么区别？

`demo-langchain.py` 用 `@tool` 装饰器把 `get_weather` 注册为工具——工具**和 Agent 运行在同一个进程里**：

```
┌─────────────────────── 单个 Python 进程 ───────────────────────┐
│  Agent (LLM)  ──直接函数调用──▶  @tool get_weather()          │
└────────────────────────────────────────────────────────────────┘
```

MCP 则把工具**拆成独立进程/独立服务**，用标准协议（JSON-RPC over stdio 或 HTTP）通信：

```
┌──────────────┐   JSON-RPC / stdio    ┌────────────────────┐
│ MCP Client   │ ────────────────────▶ │  MCP Server        │
│ (LangChain   │  1. tools/list        │  (独立子进程)       │
│  Agent 进程) │  2. tools/call        │  @mcp.tool() ×N    │
└──────────────┘ ◀──────────────────── └────────────────────┘
```

**为什么值得这么做：**

1. **跨语言 / 跨进程复用** —— Server 可以用任何语言写，一次实现，多个 Agent（Claude Desktop、Cursor、自研 Agent）共享。
2. **解耦与隔离** —— 工具崩溃不会拖垮 Agent 进程；依赖冲突互不影响。
3. **标准化生态** —— 直接接入社区现成的 Server（如本例的 `open-meteo-mcp`），无需自己写爬虫/API 封装。
4. **动态发现** —— Agent 启动时通过 `tools/list` 自动获知有哪些工具、参数是什么，不需要硬编码。

**本例的调用链：**

```
用户提问
  → LangChain create_agent（LLM 决策要不要用工具）
  → MultiServerMCPClient 把 MCP 工具适配成 LangChain BaseTool
  → stdio 子进程调用 MCP Server 的 tools/call
  → 结果回传给 LLM 总结 → 输出答案
```

---

## 二、运行前须知（重要）

| 项 | 要求 |
|---|---|
| **Python 版本** | **≥ 3.10**（`mcp` 与 `langchain-mcp-adapters` 的硬性要求；项目根目录默认的 3.8 **跑不了**） |
| **Jupyter Kernel** | 请选择 `agent` 环境（Python 3.11.15） |
| **环境变量** | `LLM_API_KEY` 必须已设置 |
| **可选环境变量** | `OPENWEATHER_API_KEY`（仅场景二的 `mcp-server-weather` 需要） |

安装依赖：

```bash
pip install "mcp>=1.24.0" "langchain-mcp-adapters>=0.3.2" open-meteo-mcp mcp-server-weather
```

> 💡 场景一的自建 Server 只依赖 `mcp`，不需要联网；场景二的 `open-meteo-mcp` 需要联网访问 Open-Meteo API。


---
# 第 0 步：环境准备

统一处理三件事：**Windows 控制台编码**、**项目路径定位**、**LLM 实例创建**。


In [ ]:
"""环境准备：编码修复 + 路径定位 + 依赖检查"""
import os
import sys
import json
import asyncio
import platform
from pathlib import Path
from configparser import ConfigParser

# ── 0.1 Windows 控制台默认 GBK，天气数据含中文/emoji，不修会 UnicodeEncodeError ──
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("PYTHONUTF8", "1")


def find_project_root(start: Path) -> Path:
    """向上查找含 config.ini 与 demo/ 的项目根目录。

    Jupyter 的 cwd 通常是 Notebook 所在目录（demo/），
    但 VS Code 有时会设成工作区根目录，因此两种情况都要兼容。
    """
    for candidate in (start, *start.parents):
        if (candidate / "config.ini").exists() and (candidate / "demo").is_dir():
            return candidate
    return start


BASE_DIR = find_project_root(Path.cwd().resolve())
DEMO_DIR = BASE_DIR / "demo"

print(f"Python      : {sys.version.split()[0]}  ({platform.python_implementation()})")
print(f"项目根目录  : {BASE_DIR}")
print(f"demo 目录   : {DEMO_DIR}")
print(f"LLM_API_KEY : {'已设置' if os.environ.get('LLM_API_KEY') else '❌ 未设置'}")

# ── 0.2 依赖版本检查（MCP 生态要求 Python >= 3.10）──
if sys.version_info < (3, 10):
    raise RuntimeError(
        f"当前 Python {sys.version_info.major}.{sys.version_info.minor} 过低，"
        "MCP 需要 >= 3.10，请切换 Jupyter Kernel 到 agent 环境。"
    )

import importlib.metadata as md

for pkg in ("mcp", "langchain-mcp-adapters", "langchain", "open-meteo-mcp", "mcp-server-weather"):
    try:
        print(f"  ✅ {pkg:<24} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  ⚠️  {pkg:<24} 未安装（pip install {pkg}）")

### 0.3 ⚠️ 关键补丁：让 MCP stdio 能在 Jupyter Kernel 里工作

**这是本 Notebook 最容易踩、也最隐蔽的坑，务必理解。**

`mcp` SDK 的 stdio 客户端签名是：

```python
# site-packages/mcp/client/stdio/__init__.py
async def stdio_client(server, errlog: TextIO = sys.stderr):
    ...
    # Windows: create_windows_process(command, args, env, errlog, cwd)
    #   → subprocess.Popen(..., stderr=errlog)
    #     → msvcrt.get_osfhandle(stderr.fileno())
```

它要把 `sys.stderr` **当成子进程的标准错误句柄**传给 `subprocess.Popen`。
在普通脚本里没问题，但在 **Jupyter Kernel** 中，`sys.stderr` 被 ipykernel 替换成了
`ipykernel.iostream.OutStream`——它**没有真正的文件描述符**，调用 `fileno()` 直接抛异常：

```
io.UnsupportedOperation: fileno
```

于是 `MultiServerMCPClient` 一连 stdio Server 就崩，而且报错栈深埋在 `subprocess.py` 里，很难联想到是 Jupyter 的锅。

**解法**：检测到 `sys.stderr` 不可用时，把 `stdio_client` 的 `errlog` 重定向到一个真实日志文件。
这样 Server 的 stderr 日志仍然完整可查（下面会读出来），同时不再依赖 Kernel 的伪 stderr。

> ✅ 在普通 `.py` 脚本 / 终端里运行时无需此补丁——下面的函数会自动检测到 stderr 可用并跳过打补丁。


In [ ]:
"""补丁：修复 Jupyter Kernel 下 MCP stdio 的 UnsupportedOperation: fileno

设计要点：
  1. 幂等 —— 重复执行不会叠加包装（用 _errlog_patched 标记）
  2. 自适应 —— stderr 本身可用时（普通脚本/终端）直接返回，不做任何修改
  3. 可观测 —— Server 的 stderr 写入真实文件，路径返回给调用方以便后续查看
"""
import functools
import tempfile


def patch_mcp_stdio_errlog(log_path: str | None = None) -> str | None:
    """让 MCP stdio 传输在没有真实 stderr 文件描述符的环境（Jupyter）中可用。

    Returns:
        补丁生效时返回 Server stderr 日志文件路径；无需补丁时返回 None。
    """
    import mcp.client.stdio as stdio_mod
    # langchain-mcp-adapters 是 `from mcp.client.stdio import stdio_client`，
    # 即在自己模块里持有了一份函数引用，所以两个模块都要替换。
    import langchain_mcp_adapters.sessions as sess_mod

    def _usable(stream) -> bool:
        try:
            fd = stream.fileno()
            return isinstance(fd, int) and fd >= 0
        except Exception:      # OutStream 抛 UnsupportedOperation
            return False

    if _usable(sys.stderr):
        print("ℹ️  sys.stderr 具备真实文件描述符，无需补丁（普通脚本/终端环境）。")
        return None

    path = log_path or str(Path(tempfile.gettempdir()) / "mcp_server_stderr.log")
    handle = open(path, "a", encoding="utf-8", errors="replace")

    def _wrap(module, attr: str = "stdio_client"):
        original = getattr(module, attr)
        if getattr(original, "_errlog_patched", False):
            return                                  # 幂等：已打过补丁

        @functools.wraps(original)
        def patched(server, errlog=None):
            return original(server, errlog if _usable(errlog) else handle)

        patched._errlog_patched = True
        setattr(module, attr, patched)

    _wrap(stdio_mod)
    _wrap(sess_mod)
    return path


MCP_STDERR_LOG = patch_mcp_stdio_errlog()

if MCP_STDERR_LOG:
    print(f"✅ 已打补丁：MCP Server 的 stderr 将写入\n   {MCP_STDERR_LOG}")
else:
    print("✅ 环境正常，跳过补丁。")

In [ ]:
"""读取 config.ini 并创建 LLM 实例（与 demo-langchain.py 保持一致）"""
from langchain_openai import ChatOpenAI

config = ConfigParser()
config.read(BASE_DIR / "config.ini", encoding="utf-8")

api_key = os.environ.get("LLM_API_KEY")
if not api_key:
    raise ValueError("环境变量 LLM_API_KEY 未设置，请先设置 API Key")

llm = ChatOpenAI(
    api_key=api_key,
    base_url=config.get("LLM", "api_url"),
    model=config.get("LLM", "model_name"),
    temperature=config.get("LLM", "temperature"),
    max_tokens=512,
)

print(f"base_url : {config.get('LLM', 'api_url')}")
print(f"model    : {config.get('LLM', 'model_name')}")
print(f"✅ LLM 实例创建成功")

In [ ]:
"""MCP 工具返回值解析辅助函数

MCP 协议的 tools/call 返回的是「内容块列表」，经 LangChain 适配后
tool.content 仍然是这种结构，而不是纯字符串：

    [{'type': 'text', 'text': '{"城市": "北京", ...}', 'id': 'lc_xxx'}]

直接丢给 print 很难读，所以统一抽取其中的文本。
"""
from langchain_core.messages import ToolMessage


def extract_text(content) -> str:
    """把 MCP 内容块列表 / 字符串统一转成可读文本。"""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                parts.append(str(block.get("text", block)))
            else:
                parts.append(str(block))
        return "\n".join(parts)
    return str(content)


def print_message_trace(messages, max_len: int = 300) -> None:
    """打印 Agent 一轮完整的消息轨迹：LLM 决策 → 工具调用 → 工具结果 → 最终回答。

    这是理解 Agent-Loop 最直观的方式。
    """
    for msg in messages:
        kind = getattr(msg, "type", msg.__class__.__name__)
        tool_calls = getattr(msg, "tool_calls", None)

        if tool_calls:
            for tc in tool_calls:
                print(f"  🔧 [LLM 决策调用工具] {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")
        elif isinstance(msg, ToolMessage):
            print(f"  📦 [工具返回] {msg.name} → {extract_text(msg.content)[:max_len]}")
        elif kind == "ai" and msg.content:
            print(f"\n🤖 [Agent 最终回答]\n{str(msg.content)[:2000]}")
        elif kind == "human":
            print(f"👤 [用户] {str(msg.content)[:200]}")

---
# 场景一：自建 Weather MCP Server

仿照 `demo-langchain.py` 的 `@tool get_weather`，把同样的 Mock 天气能力**改造成一个独立的 MCP Server 进程**。

## 核心：`FastMCP` 与 `@mcp.tool()`

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("self-built-weather-server")

@mcp.tool()
def get_weather(city: str) -> str:
    """查询指定城市的天气信息"""
    ...

mcp.run(transport="stdio")
```

**对照理解（和 LangChain `@tool` 几乎一一对应）：**

| LangChain `@tool` | MCP `@mcp.tool()` |
|---|---|
| 类型注解 `city: str` → args schema | 类型注解 `city: str` → inputSchema |
| docstring → 工具描述 | docstring → 工具描述 |
| 同进程直接调用 | 跨进程 JSON-RPC 调用 |
| 注册到 `create_agent(tools=[...])` | 注册到 Server，Client 用 `tools/list` 发现 |

## ⚠️ stdio 模式头号大坑：`print()` 会污染协议通道

stdio 传输下，**stdout 被 JSON-RPC 报文独占**。Server 里任何一句 `print()` 都会插进协议流，
导致 Client 解析失败（典型报错：`JSONDecodeError` / 握手超时）。

✅ 正确做法：**日志一律写 stderr**（下面的代码用 `logging.basicConfig(stream=sys.stderr)`）。

下面的单元格会把 Server 源码写入 `demo/weather_mcp_server.py`。


In [ ]:
%%writefile weather_mcp_server.py
"""
自建 Weather MCP Server（仿照 demo-langchain.py 的 get_weather 工具）

核心概念：
- MCP Server：把「工具」独立成一个进程，通过 MCP 协议对外暴露
- FastMCP：官方高层封装，用 @mcp.tool() 装饰器把普通 Python 函数注册为 MCP 工具，
  函数签名（类型注解）+ docstring 会自动转换成 LLM 可读的 JSON Schema
- stdio 传输：Server 作为子进程启动，Client 通过标准输入/输出收发 JSON-RPC 消息

独立运行（手动验证协议握手）：
    python demo/weather_mcp_server.py

⚠️ stdio 模式下 stdout 被 MCP 协议独占，任何 print() 都会破坏 JSON-RPC 报文。
   因此日志一律输出到 stderr。
"""

import sys
import json
import logging

from mcp.server.fastmcp import FastMCP

# Windows 控制台默认 GBK，天气数据含中文与 emoji，统一切到 UTF-8 并容错，避免编码崩溃
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")

# 日志只能走 stderr（stdout 是 MCP 协议通道）
logging.basicConfig(
    level=logging.INFO,
    stream=sys.stderr,
    format="[weather-mcp] %(message)s",
)
logger = logging.getLogger("weather-mcp")

# 1. 创建 MCP Server 实例（name 会在握手时告知 Client）
mcp = FastMCP("self-built-weather-server")

# Mock 天气数据（与 demo-langchain.py 的 get_weather 保持一致）
MOCK_WEATHER = {
    "北京": {"天气": "晴", "温度": "25°C", "湿度": "40%", "风力": "微风"},
    "上海": {"天气": "多云", "温度": "22°C", "湿度": "65%", "风力": "东南风3级"},
    "广州": {"天气": "小雨", "温度": "28°C", "湿度": "80%", "风力": "南风2级"},
    "深圳": {"天气": "阴", "温度": "26°C", "湿度": "70%", "风力": "东风2级"},
}

UNKNOWN = {"天气": "未知", "温度": "N/A", "湿度": "N/A", "风力": "N/A"}


# 2. 注册工具：类型注解 -> 参数 Schema，docstring -> 工具描述
@mcp.tool()
def get_weather(city: str) -> str:
    """查询指定城市的天气信息（Mock 数据）。

    Args:
        city: 城市名称，如：北京、上海、广州、深圳
    """
    data = MOCK_WEATHER.get(city, UNKNOWN)
    result = {"城市": city, "数据源": "self-built-mcp(mock)", **data}
    logger.info("get_weather called: city=%s hit=%s", city, city in MOCK_WEATHER)
    return json.dumps(result, ensure_ascii=False)


@mcp.tool()
def list_supported_cities() -> str:
    """列出当前天气服务支持查询的所有城市名称。"""
    cities = list(MOCK_WEATHER.keys())
    logger.info("list_supported_cities called: %d cities", len(cities))
    return json.dumps(
        {"支持城市": cities, "数量": len(cities)},
        ensure_ascii=False,
    )


@mcp.tool()
def compare_weather(city_a: str, city_b: str) -> str:
    """对比两个城市的天气，返回温度更高、湿度更大的城市。

    Args:
        city_a: 第一个城市名称
        city_b: 第二个城市名称
    """
    a = MOCK_WEATHER.get(city_a)
    b = MOCK_WEATHER.get(city_b)

    if a is None or b is None:
        missing = [c for c, d in ((city_a, a), (city_b, b)) if d is None]
        return json.dumps(
            {"error": f"不支持的城市: {missing}", "支持城市": list(MOCK_WEATHER.keys())},
            ensure_ascii=False,
        )

    def _num(text: str) -> float:
        """从 '25°C' / '40%' 中取出数字部分"""
        digits = "".join(ch for ch in text if ch.isdigit() or ch == ".")
        return float(digits) if digits else 0.0

    hotter = city_a if _num(a["温度"]) >= _num(b["温度"]) else city_b
    wetter = city_a if _num(a["湿度"]) >= _num(b["湿度"]) else city_b

    logger.info("compare_weather called: %s vs %s", city_a, city_b)
    return json.dumps(
        {
            city_a: a,
            city_b: b,
            "更温暖": hotter,
            "更潮湿": wetter,
        },
        ensure_ascii=False,
    )


if __name__ == "__main__":
    logger.info("tools registered: get_weather, list_supported_cities, compare_weather")
    # 3. 以 stdio 模式启动：阻塞运行，等待 Client 通过标准输入发送 JSON-RPC 请求
    mcp.run(transport="stdio")


### 1.1 定位 Server 脚本并验证它能独立启动

MCP Server 是一个**普通可执行脚本**，可以脱离 LangChain 单独运行做冒烟测试。


In [ ]:
"""定位 Server 脚本路径（兼容 Jupyter cwd 为 demo/ 或工作区根目录两种情况）"""
SERVER_SCRIPT = None
for candidate in (
    Path("weather_mcp_server.py"),
    DEMO_DIR / "weather_mcp_server.py",
    Path.cwd() / "weather_mcp_server.py",
):
    if candidate.exists():
        SERVER_SCRIPT = candidate.resolve()
        break

if SERVER_SCRIPT is None:
    raise FileNotFoundError("未找到 weather_mcp_server.py，请先运行上一个 %%writefile 单元格")

print(f"✅ Server 脚本: {SERVER_SCRIPT}")
print(f"   大小: {SERVER_SCRIPT.stat().st_size} bytes")

In [ ]:
"""冒烟测试：不经过 LLM，直接用 MCP Client 连接自建 Server

演示 MCP 的两个核心动作：
  1. tools/list  —— 发现 Server 提供了哪些工具（动态发现，无需硬编码）
  2. tools/call  —— 直接调用某个工具，拿到底层原始返回
"""
from langchain_mcp_adapters.client import MultiServerMCPClient

# stdio 连接配置：告诉 Client「用什么命令、什么参数」把 Server 拉起来
SELF_BUILT_CONN = {
    "self_built_weather": {
        "transport": "stdio",
        "command": sys.executable,          # 用当前 Notebook 的同一个 Python 解释器
        "args": [str(SERVER_SCRIPT)],       # 脚本路径
        # "env": {...},                     # 可选：为子进程定制环境变量
    }
}


async def probe_self_built():
    client = MultiServerMCPClient(SELF_BUILT_CONN)

    # ── 1) 工具发现：Client 启动子进程 → 握手 → 发送 tools/list ──
    tools = await client.get_tools()
    print(f"发现 {len(tools)} 个 MCP 工具（已由 MCP 协议自动适配为 LangChain BaseTool）:\n")
    for t in tools:
        first_line = (t.description or "").strip().splitlines()[0]
        print(f"  • {t.name:<24} {first_line}")
        print(f"    args schema: {json.dumps(t.args, ensure_ascii=False)}")

    # ── 2) 直接调用工具：不经过 LLM，看清 MCP 的原始返回结构 ──
    print("\n" + "=" * 70)
    get_weather = next(t for t in tools if t.name == "get_weather")

    raw = await get_weather.ainvoke({"city": "北京"})
    print("原始返回类型 :", type(raw).__name__)
    print("原始返回内容 :", json.dumps(raw, ensure_ascii=False)[:300])
    print("抽取文本后   :", extract_text(raw))

    print("\n--- 未收录城市（走 Mock 的兜底分支）---")
    print(extract_text(await get_weather.ainvoke({"city": "成都"})))

    print("\n--- 多参数工具 compare_weather ---")
    compare = next(t for t in tools if t.name == "compare_weather")
    print(extract_text(await compare.ainvoke({"city_a": "北京", "city_b": "广州"})))


await probe_self_built()

In [ ]:
"""查看 Server 端日志：MCP Server 的 stderr 输出去了哪里？

Server 里的 logging 写的是 stderr（不是 stdout，避免污染协议通道）。
在 Jupyter 中这些日志不会显示在单元格输出里，而是被上面的补丁重定向到文件。
"""
if MCP_STDERR_LOG:
    log_text = Path(MCP_STDERR_LOG).read_text(encoding="utf-8", errors="replace")
    tail = log_text.strip().splitlines()[-20:]
    print(f"📄 {MCP_STDERR_LOG}（最后 {len(tail)} 行）\n" + "=" * 70)
    print("\n".join(tail))
else:
    print("ℹ️  Server 日志直接输出到了当前进程的 stderr（终端环境），无需查看文件。")

### 1.2 把 MCP 工具接入 LangChain Agent

这一步和 `demo-langchain.py` **完全一样**——`create_agent(tools=[...])`。

区别只在于：`tools` 不再来自本地 `@tool` 函数，而是来自 `client.get_tools()` 的动态发现结果。

> 📌 **关键机制**：`get_tools()` 返回的每个工具**在被调用时都会新建一个 MCP session**（源码注释：*"A new session will be created for each tool call"*）。
> 好处是无状态、易并发；代价是每次调用都有进程/握手开销，高频场景可考虑 `streamable-http` 长连接。


In [ ]:
"""创建 Agent（MCP 工具版）——对比 demo-langchain.py，只有 tools 的来源不同"""
from langchain.agents import create_agent

SYSTEM_PROMPT = (
    "你是一个智能助手，可以使用工具来帮助用户完成任务。"
    "调用工具后请基于工具返回的真实数据作答，不要编造。"
)


async def build_agent(connections: dict, system_prompt: str = SYSTEM_PROMPT):
    """通用工厂：给定 MCP 连接配置 → 返回 (agent, tools)。

    场景一 / 场景二 / 多 Server 联合都复用这个函数，
    这正是 MCP 的价值所在：换 Server 不用改 Agent 代码。
    """
    client = MultiServerMCPClient(connections)
    tools = await client.get_tools()
    agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)
    return agent, tools


async def ask(agent, question: str, recursion_limit: int = 10):
    """单轮提问，并打印完整 Agent-Loop 轨迹。"""
    print(f"👤 [用户] {question}\n" + "-" * 70)
    result = await agent.ainvoke(
        {"messages": [("user", question)]},
        config={"recursion_limit": recursion_limit},   # 控制最大迭代次数
    )
    print_message_trace(result["messages"])
    return result


agent_self, tools_self = await build_agent(SELF_BUILT_CONN)
print(f"✅ Agent 创建成功，已挂载 {len(tools_self)} 个 MCP 工具: {[t.name for t in tools_self]}")

In [ ]:
"""单工具调用：最基础的 MCP 链路"""
await ask(agent_self, "北京今天天气怎么样？")

In [ ]:
"""多工具编排：LLM 自主决定先列城市、再逐个查询、最后汇总

观察下方轨迹里 LLM 发起了几次 tool_call —— 这就是 Agent-Loop 在 MCP 上的体现。
"""
await ask(agent_self, "你的天气服务支持哪些城市？请分别查询它们的天气，然后告诉我哪个城市最适合出门散步。")

In [ ]:
"""多参数工具 + 对比推理"""
await ask(agent_self, "北京和广州比，哪个更热、哪个更潮湿？")

---
# 场景二：连接开源天气 MCP Server

不写一行工具代码，直接接入社区现成的 MCP Server。这里演示两个，可按需切换。

## 2.1 `open-meteo-mcp`（推荐：真实数据 + 免 API Key）

| 项 | 说明 |
|---|---|
| PyPI | `open-meteo-mcp` v0.2.0（要求 Python ≥ 3.10） |
| 源码 | https://github.com/isamauny/open_meteo_mcp |
| 数据源 | [Open-Meteo](https://open-meteo.com/) 免费公开气象 API，**无需注册/无需 Key** |
| 启动方式 | `python -m open_meteo_mcp`（stdio 默认模式） |
| 传输模式 | stdio / sse / streamable-http 均支持 |
| 工具数 | **8 个** |

提供的工具（由 `tools/list` 动态发现，注意 `get_weather_byDateTimeRange` 是驼峰命名）：

1. `get_current_weather` — 当前天气（温度/体感/湿度/露点/风向风速阵风/降水/气压/云量/UV/能见度）
2. `get_weather_byDateTimeRange` — 指定日期区间的逐小时天气
3. `get_weather_details` — 结构化 JSON 天气数据
4. `get_air_quality` — 空气质量（PM2.5/PM10/O₃/NO₂/CO/SO₂ + 健康建议）
5. `get_air_quality_details` — 结构化 JSON 空气质量数据
6. `get_current_datetime` — 任意时区当前时间
7. `get_timezone_info` — 时区信息
8. `convert_time` — 时区时间换算

> ⚠️ **城市名必须用英文**（内部走 Open-Meteo Geocoding API），例如 `Beijing` / `Shanghai` / `Tokyo`。

## 2.2 `mcp-server-weather`（需要 OpenWeather API Key）

| 项 | 说明 |
|---|---|
| PyPI | `mcp-server-weather` v0.1.4（要求 Python ≥ 3.10） |
| 数据源 | OpenWeather `/data/2.5/weather`，**需自备 API Key**（免费注册：https://openweathermap.org/api） |
| 启动方式 | 控制台脚本 `mcp-server-weather --api_key <KEY>`（**没有** `python -m` 入口） |
| 工具数 | **1 个**：`query_weather(city)` — 城市英文名，返回中文格式化天气 |

## 2.3 切换开关

只需修改下面的 `USE_OPENTE_METEO` / `USE_OPENWEATHER` 两个布尔值即可自由组合。


In [ ]:
"""开源 MCP Server 的连接配置与开关"""
# ══════════ 开关：想连哪个开源 Server 就打开哪个 ══════════
USE_OPEN_METEO = True      # open-meteo-mcp：真实数据，免 API Key（推荐）
USE_OPENWEATHER = False    # mcp-server-weather：需要 OPENWEATHER_API_KEY

# ── open-meteo-mcp：用 `python -m open_meteo_mcp` 启动（stdio 是默认模式）──
OPEN_METEO_CONN = {
    "open_meteo": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["-m", "open_meteo_mcp"],
    }
}

# ── mcp-server-weather：只能通过 pip 装出来的控制台脚本启动，Key 走命令行参数 ──
OPENWEATHER_API_KEY = os.environ.get("OPENWEATHER_API_KEY", "")


def _find_mcp_server_weather_exe() -> str | None:
    """定位 pip 安装的 mcp-server-weather 控制台脚本。

    该包没有 __main__.py，无法用 `python -m` 启动，
    只能调用 Scripts/（Windows）或 bin/（*nix）下的可执行文件。
    """
    script_dir = Path(sys.executable).parent
    for rel in (("Scripts", "mcp-server-weather.exe"), ("bin", "mcp-server-weather")):
        candidate = script_dir.joinpath(*rel)
        if candidate.exists():
            return str(candidate)
    return None


MSW_EXE = _find_mcp_server_weather_exe()

OPENWEATHER_CONN = {
    "openweathermap": {
        "transport": "stdio",
        "command": MSW_EXE or "mcp-server-weather",
        "args": ["--api_key", OPENWEATHER_API_KEY or "MISSING_KEY"],
    }
}

print("开源 MCP Server 配置：")
print(f"  USE_OPEN_METEO  = {USE_OPEN_METEO}")
print(f"  USE_OPENWEATHER = {USE_OPENWEATHER}")
print(f"  open-meteo 启动命令      : {sys.executable} -m open_meteo_mcp")
print(f"  mcp-server-weather 脚本  : {MSW_EXE or '❌ 未找到（pip install mcp-server-weather）'}")
print(f"  OPENWEATHER_API_KEY      : {'已设置' if OPENWEATHER_API_KEY else '⚠️  未设置（该 Server 会返回错误提示）'}")

### 2.1 探测 `open-meteo-mcp`：发现 8 个工具并取回**真实气象数据**

In [ ]:
"""连接开源 open-meteo-mcp：工具发现 + 真实数据调用"""
if not USE_OPEN_METEO:
    print("⏭️  已跳过（USE_OPEN_METEO = False）")
else:
    async def probe_open_meteo():
        client = MultiServerMCPClient(OPEN_METEO_CONN)
        tools = await client.get_tools()

        print(f"✅ 从开源 MCP Server 动态发现 {len(tools)} 个工具：\n")
        for t in tools:
            first_line = (t.description or "").strip().splitlines()[0]
            print(f"  • {t.name:<30} {first_line[:60]}")

        print("\n" + "=" * 70)
        current = next(t for t in tools if t.name == "get_current_weather")

        # ⚠️ 城市名必须英文：内部走 Open-Meteo Geocoding API
        print("【真实数据】get_current_weather(city='Beijing'):\n")
        print(extract_text(await current.ainvoke({"city": "Beijing"})))

        print("\n【真实数据】get_air_quality(city='Shanghai'):\n")
        air = next(t for t in tools if t.name == "get_air_quality")
        print(extract_text(await air.ainvoke({"city": "Shanghai"}))[:600])

    await probe_open_meteo()

### 2.2 用开源 MCP 工具驱动 Agent

注意：这一格的代码和场景一**几乎完全相同**——只是把连接配置换成了 `OPEN_METEO_CONN`。
换 Server 不改 Agent 逻辑，这就是 MCP 标准化的收益。


In [ ]:
"""创建接入开源天气 MCP 的 Agent，并进行真实天气问答"""
if not USE_OPEN_METEO:
    print("⏭️  已跳过（USE_OPEN_METEO = False）")
else:
    agent_om, tools_om = await build_agent(
        OPEN_METEO_CONN,
        system_prompt=(
            "你是一个气象助手，可以使用工具查询真实天气与空气质量数据。"
            "注意：调用天气工具时城市名必须使用英文（如 Beijing、Shanghai、Tokyo）。"
            "回答时请把关键指标整理清晰，并用中文说明。"
        ),
    )
    print(f"✅ Agent 创建成功，挂载 {len(tools_om)} 个开源 MCP 工具")
    print(f"   {[t.name for t in tools_om]}\n")

    await ask(agent_om, "北京现在的天气怎么样？适合出门跑步吗？")

In [ ]:
"""跨工具编排：天气 + 空气质量 + 时区，LLM 自主选择多个 MCP 工具"""
if not USE_OPEN_METEO:
    print("⏭️  已跳过（USE_OPEN_METEO = False）")
else:
    await ask(
        agent_om,
        "帮我对比一下东京和纽约现在的天气与空气质量，并告诉我两地现在分别是几点。",
        recursion_limit=20,   # 多工具编排需要更多迭代轮次
    )

### 2.3 探测 `mcp-server-weather`（需要 OpenWeather API Key）

In [ ]:
"""连接 mcp-server-weather：需要 OPENWEATHER_API_KEY

未配置 Key 时该 Server 不会崩溃，而是优雅返回「API_KEY 未设置」的提示，
因此这一格在任何情况下都能安全运行，方便观察两种结果的差异。
"""
if not USE_OPENWEATHER:
    print("⏭️  已跳过（USE_OPENWEATHER = False）")
elif MSW_EXE is None:
    print("❌ 未找到 mcp-server-weather 可执行文件，请先：pip install mcp-server-weather")
else:
    async def probe_openweather():
        client = MultiServerMCPClient(OPENWEATHER_CONN)
        tools = await client.get_tools()

        print(f"✅ 发现 {len(tools)} 个工具：")
        for t in tools:
            print(f"  • {t.name:<16} args={json.dumps(t.args, ensure_ascii=False)}")
            print(f"    描述: {(t.description or '').strip()}")

        print("\n" + "=" * 70)
        if not OPENWEATHER_API_KEY:
            print("⚠️  OPENWEATHER_API_KEY 未设置 —— 下面会看到 Server 返回的错误提示（而非崩溃）：\n")

        print(extract_text(await tools[0].ainvoke({"city": "Beijing"})))

    await probe_openweather()

In [ ]:
"""用 mcp-server-weather 驱动 Agent（需已配置 OPENWEATHER_API_KEY）"""
if not USE_OPENWEATHER:
    print("⏭️  已跳过（USE_OPENWEATHER = False）")
elif MSW_EXE is None:
    print("❌ 未找到 mcp-server-weather 可执行文件")
elif not OPENWEATHER_API_KEY:
    print("⚠️  已跳过：请先设置环境变量 OPENWEATHER_API_KEY，然后把 USE_OPENWEATHER 改为 True")
else:
    agent_ow, tools_ow = await build_agent(
        OPENWEATHER_CONN,
        system_prompt=(
            "你是一个天气助手，使用工具查询真实天气。"
            "城市名请使用英文。请用中文清晰汇总结果。"
        ),
    )
    await ask(agent_ow, "伦敦今天天气如何？需要带伞吗？")

---
# 场景三（进阶）：同时连接多个 MCP Server

`MultiServerMCPClient` 的名字里带 **Multi**，正是为此设计：一次配置多个 Server，
`get_tools()` 会把所有 Server 的工具**聚合成一个扁平列表**交给 Agent。

下面把「自建 Mock Server」和「开源真实数据 Server」同时挂上，
让 LLM 自己判断该用哪个数据源——注意观察它在同一轮对话里跨越两个进程调用工具。

> 💡 若不同 Server 存在同名工具，可在 `MultiServerMCPClient(..., tool_name_prefix=True)`
> 开启前缀命名（如 `self_built_weather_get_weather`）来避免冲突。


In [ ]:
"""多 Server 联合：自建 Mock + 开源真实数据，同时挂载"""
MULTI_CONN = {**SELF_BUILT_CONN}
if USE_OPEN_METEO:
    MULTI_CONN.update(OPEN_METEO_CONN)

print("本次将连接以下 MCP Server：")
for name, conn in MULTI_CONN.items():
    print(f"  • {name:<20} → {conn['command']} {' '.join(conn['args'])}")

agent_multi, tools_multi = await build_agent(
    MULTI_CONN,
    system_prompt=(
        "你是一个智能天气助手，同时接入了两个数据源：\n"
        "1. 自建 Mock 天气服务：工具 get_weather / list_supported_cities / compare_weather，"
        "只支持北京、上海、广州、深圳，返回的是演示用假数据；\n"
        "2. 开源 Open-Meteo 服务：工具 get_current_weather 等，覆盖全球城市，返回真实数据，城市名需英文。\n"
        "请根据用户问的城市自行选择合适的工具，并在回答中说明数据来源（Mock 还是真实数据）。"
    ),
)

print(f"\n✅ 聚合了 {len(tools_multi)} 个工具（来自 {len(MULTI_CONN)} 个 MCP Server）：")
for t in tools_multi:
    print(f"  • {t.name}")

In [ ]:
"""跨 Server 编排：一次对话里同时用到 Mock 数据与真实数据"""
await ask(
    agent_multi,
    "先用 Mock 服务查一下深圳的天气，再用真实数据源查一下深圳现在的实际天气，"
    "对比一下两者差异，并说明哪一个是真实的。",
    recursion_limit=20,
)

---
# 场景四：多轮对话（带历史记忆）

与 `demo-langchain.py` 的交互循环等价，只是工具来自 MCP。
运行下面的单元格后会进入交互模式，输入 `exit` / `quit` / `clear` 控制。


In [ ]:
"""多轮对话循环：手动维护 chat_history，把完整消息列表回传给 Agent

与 demo-langchain.py 的差异仅在于 tools 来自 MCP Server。
MCP 工具是无状态的，对话记忆完全由客户端（这里的 chat_history）负责。
"""
from langchain_core.messages import HumanMessage

EXIT_WORDS = {"quit", "exit", "q", "byebye", "拜拜", "再见"}


def _stdin_available() -> bool:
    """检测当前前端是否支持 input()。

    VS Code / JupyterLab 交互执行时内核的 _allow_stdin 为 True；
    但 nbclient / nbconvert / CI 等无人值守执行时为 False，
    此时调用 input() 会抛 StdinNotImplementedError。
    """
    try:
        kernel = get_ipython().kernel          # noqa: F821 (IPython 注入)
        return bool(getattr(kernel, "_allow_stdin", False))
    except Exception:
        return False


async def run_turn(agent, chat_history: list, user_input: str) -> list:
    """执行一轮对话，返回更新后的完整消息历史。

    交互模式和演示模式共用这段逻辑，保证两者行为一致。
    """
    chat_history.append(HumanMessage(content=user_input))

    response = await agent.ainvoke(
        {"messages": chat_history},
        config={"recursion_limit": 15},
    )
    # 用 Agent 返回的完整消息列表覆盖历史（含 tool_call / ToolMessage）
    # 这正是多轮记忆的载体：工具调用记录也一并留在历史里
    return response["messages"]


async def chat_loop(agent, max_turns: int = 10):
    """交互式多轮对话。max_turns 防止在 Notebook 里无限阻塞。"""
    chat_history = []
    print("=" * 60)
    print("MCP Agent 多轮对话（输入 exit/quit 退出，clear 清空历史）")
    print(f"最多 {max_turns} 轮，避免 Notebook 长时间阻塞")
    print("=" * 60)

    for _ in range(max_turns):
        try:
            user_input = input("\n用户: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n结束对话。")
            break

        if user_input.lower() in EXIT_WORDS:
            print("拜拜！祝您有美好的一天！")
            break
        if not user_input:
            print("请输入有效指令或问题。")
            continue
        if user_input.lower() == "clear":
            chat_history = []
            print("🧹 对话历史已清空。")
            continue

        chat_history = await run_turn(agent, chat_history, user_input)
        print(f"\nAgent: {chat_history[-1].content}")

    return chat_history


async def demo_multi_turn(agent, questions: list[str]):
    """无人值守模式：用预置问题演示多轮对话与历史记忆。"""
    chat_history = []
    for i, q in enumerate(questions, 1):
        print(f"\n{'=' * 70}\n【第 {i} 轮】\n👤 用户: {q}\n{'-' * 70}")
        chat_history = await run_turn(agent, chat_history, q)
        print(f"🤖 Agent: {chat_history[-1].content}")
        print(f"\n   📊 当前历史累计 {len(chat_history)} 条消息"
              f"（{sum(1 for m in chat_history if m.type == 'tool')} 条为工具返回）")
    return chat_history


# ── 根据执行环境自动选择模式 ──
# 默认用「多 Server 联合」的 Agent；想换成别的把 agent_multi 改掉即可
if _stdin_available():
    history = await chat_loop(agent_multi, max_turns=5)
else:
    print("ℹ️  当前前端不支持 input()（如 nbclient / nbconvert / CI 无人值守执行）。")
    print("   自动切换为「预置问题演示模式」。在 VS Code / JupyterLab 中手动运行时可正常交互。\n")
    history = await demo_multi_turn(agent_multi, [
        "深圳的天气怎么样？",           # 命中 Mock Server（深圳在 Mock 列表里）
        "那它和广州比谁更潮湿？",        # 依赖上一轮语境 → 验证多轮记忆
        "顺便看看东京现在的真实天气。",   # 跨到开源真实数据 Server
    ])

print(f"\n📊 对话结束，累计 {len(history)} 条消息。")

---
# 总结

## 一、三种写法的演进关系

| Demo 文件 | 工具形态 | Agent-Loop 实现 | 工具位置 |
|---|---|---|---|
| `demo.py` | 手写注册表 + OpenAI tools schema | 自己写 `for` 循环 | 同进程 |
| `demo-langchain.py` | `@tool` 装饰器 | `create_agent` 自动 | 同进程 |
| `demo-langgraph.py` | `@tool` + `ToolNode` | `StateGraph` 显式建图 | 同进程 |
| **本 Notebook** | **`@mcp.tool()` + MCP 协议** | **`create_agent` 自动** | **独立子进程** |

**关键认知：对 LangChain Agent 来说，MCP 工具和本地 `@tool` 工具是完全等价的 `BaseTool`。**
`langchain-mcp-adapters` 做的就是把 MCP 的 `tools/list` + `tools/call` 适配成 LangChain 工具接口，
所以 Agent 侧代码（`create_agent` / `ainvoke` / 消息历史管理）一行都不用改。

## 二、代码骨架回顾

```python
# Server 侧（weather_mcp_server.py）
mcp = FastMCP("name")

@mcp.tool()
def get_weather(city: str) -> str:
    """查询天气"""
    ...

mcp.run(transport="stdio")

# Client 侧（Notebook）
client = MultiServerMCPClient({
    "server_name": {"transport": "stdio", "command": sys.executable, "args": [script]},
})
tools = await client.get_tools()                       # 动态发现
agent = create_agent(model=llm, tools=tools, system_prompt="...")
result = await agent.ainvoke({"messages": [("user", "...")]})
```

## 三、踩坑清单（都已在本 Notebook 中处理）

| # | 坑 | 现象 | 解法 |
|---|---|---|---|
| 1 | **Python 版本** | `mcp` / `langchain-mcp-adapters` 装不上或导入失败 | 必须 **≥ 3.10**（本例用 3.11.15 的 `agent` 环境） |
| 2 | **⭐ Jupyter 下 stdio 崩溃** | `io.UnsupportedOperation: fileno`（栈埋在 `subprocess.py`） | ipykernel 的 `sys.stderr` 无真实文件描述符，需重定向 `stdio_client` 的 `errlog`（见 0.3 补丁） |
| 3 | **stdio 下 `print()`** | Client 端 `JSONDecodeError`、握手超时 | Server 日志全部写 **stderr** |
| 4 | **Windows GBK 控制台** | `UnicodeEncodeError: 'gbk' codec can't encode '\u26a0'` | `sys.stdout.reconfigure(encoding="utf-8", errors="replace")` |
| 5 | **返回值不是字符串** | 打印出 `[{'type': 'text', 'text': ...}]` 很乱 | 用 `extract_text()` 抽取内容块文本 |
| 6 | **每次调用新建 session** | 高频调用时开销明显 | 无状态换便利；高频场景改用 `streamable-http` 长连接 |
| 7 | **异步 API** | `get_tools()` / `ainvoke()` 忘了 `await` 拿到协程对象 | Notebook 顶层可直接 `await`；脚本里用 `asyncio.run()` |
| 8 | **开源 Server 启动方式各异** | `mcp-server-weather` 无 `python -m` 入口 | 它只能用 pip 装出的**控制台脚本**启动；`open-meteo-mcp` 用 `python -m open_meteo_mcp` |
| 9 | **城市名语言** | 查不到城市 / 返回空 | Open-Meteo 与 OpenWeather 都要求**英文城市名** |
| 10 | **多工具编排被截断** | `GraphRecursionError` | 调大 `config={"recursion_limit": N}` |
| 11 | **同名工具冲突** | 多 Server 时工具互相覆盖 | `MultiServerMCPClient(..., tool_name_prefix=True)` 加 Server 名前缀 |
| 12 | **无人值守执行时 `input()`** | `StdinNotImplementedError`（nbclient / nbconvert / CI） | 检查 `get_ipython().kernel._allow_stdin`，不支持时降级为预置问题演示模式 |

> 🔎 **坑 #2 的排查心法**：报错栈最后一行是 `msvcrt.get_osfhandle(stderr.fileno())`，
> 看起来像是「MCP Server 起不来」，实际是「Client 没法把 Jupyter 的伪 stderr 传给子进程」。
> 只要记住 **stdio 传输需要一个真实的 OS 文件句柄**，就能快速定位。

## 四、延伸阅读

- MCP 官方规范：https://modelcontextprotocol.io
- Python SDK（`FastMCP`）：https://github.com/modelcontextprotocol/python-sdk
- LangChain 适配器：https://github.com/langchain-ai/langchain-mcp-adapters
- `open-meteo-mcp`：https://github.com/isamauny/open_meteo_mcp
- Open-Meteo API（免费无 Key）：https://open-meteo.com

## 五、下一步可以试什么

1. 把自建 Server 的传输方式从 `stdio` 换成 `streamable-http`，用 `mcp.run(transport="streamable-http")` 起服务，Client 侧改成 `{"transport": "streamable_http", "url": "http://127.0.0.1:8000/mcp"}`。
2. 给自建 Server 加 `@mcp.resource()`（暴露数据）和 `@mcp.prompt()`（暴露提示词模板），体验 MCP 除 tools 之外的另两类原语。
3. 把本项目的 `core/tools/custom/` 下的 Skill 包装成 MCP Server，让外部 Agent（Claude Desktop / Cursor）也能调用。
